## ADR-Influenced Elo — Analysis Notebook

Production model: ADR-influenced Elo with strict timeline correctness.

1. **Setup** — Tunables, helpers, data loading.
2. **Baseline run** — Summary charts for the default configuration.
3. **Sensitivity analysis** — How K-factor and ADR parameters shape the rating distribution.

In [28]:
import sys, pathlib
_src = str(pathlib.Path(__file__).resolve().parent.parent) if "__file__" in dir() else str(pathlib.Path.cwd().parent)
if _src not in sys.path:
    sys.path.insert(0, _src)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from db.connection_db import get_conn

# Default tuning 
TUNE = {
    "K_FACTOR": 24.0,
    "BASE_RATING": 1500.0,
    "ADR_ALPHA": 0.35,
    "ADR_SPREAD": 15.0,
    "ADR_MIN_MULT": 0.85,
    "ADR_MAX_MULT": 1.15,
    "ADR_PRIOR_MATCHES": 5.0,
    "INITIAL_GLOBAL_ANCHOR": 80.0,
}

# elpers 
def table_columns(conn, table_name: str) -> set[str]:
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    return {str(r[1]) for r in rows}

def first_existing(columns: set[str], candidates: list[str], label: str) -> str:
    for c in candidates:
        if c in columns:
            return c
    raise RuntimeError(f"No usable {label} column. Tried: {candidates}")

def _match_sort_key(value):
    txt = str(value)
    return (0, int(txt)) if txt.isdigit() else (1, txt)

def _clip(x, lo, hi):
    return max(lo, min(hi, x))

def _normalize_team_text(value):
    return str(value or "").strip()

def _is_canonical_team(value):
    return _normalize_team_text(value).lower() in {"teama", "teamb"}

def _canonical_label(value):
    txt = _normalize_team_text(value).lower()
    if txt == "teama": return "TeamA"
    if txt == "teamb": return "TeamB"
    return _normalize_team_text(value)

def _build_team_mapping_for_match(group):
    names = []
    for raw in group["team_name"].tolist():
        t = _normalize_team_text(raw)
        if not t or t.lower() == "all":
            continue
        if t not in names:
            names.append(t)
    if set(t.lower() for t in names).issubset({"teama", "teamb"}):
        return {t: _canonical_label(t) for t in names}
    if len(names) >= 2:
        return {names[0]: "TeamA", names[1]: "TeamB"}
    if len(names) == 1:
        return {names[0]: "TeamA"}
    return {}

def _map_team_name(match_id, team_name, team_map_by_match):
    team_raw = _normalize_team_text(team_name)
    if not team_raw: return team_raw
    if _is_canonical_team(team_raw): return _canonical_label(team_raw)
    return team_map_by_match.get(match_id, {}).get(team_raw, team_raw)

def _map_winner_name(match_id, winner_name, team_map_by_match):
    winner_raw = _normalize_team_text(winner_name)
    if not winner_raw: return winner_raw
    if _is_canonical_team(winner_raw): return _canonical_label(winner_raw)
    return team_map_by_match.get(match_id, {}).get(winner_raw, winner_raw)

print("Setup loaded.")

Setup loaded.


In [29]:
# Load outcome + ADR datasets 
with get_conn() as conn:
    mps_cols = table_columns(conn, "match_player_stats")
    m_cols   = table_columns(conn, "matches")
    mm_cols  = table_columns(conn, "match_maps")

    steam_col = first_existing(mps_cols, ["steamid64", "steam64_id"], "steam id")
    name_col  = first_existing(mps_cols, ["name", "player_name"], "player name")
    team_col  = first_existing(mps_cols, ["team", "team_name"], "team")

    if "match_id" in m_cols:
        preferred = ["match_id", "match_time", "start_time", "end_time",
                      "created_at", "finished_at", "winner"]
        match_cols = [c for c in preferred if c in m_cols]
        matchhistory_df = pd.read_sql_query(
            f"SELECT {', '.join(match_cols)} FROM matches ORDER BY match_id ASC", conn)
    else:
        matchhistory_df = pd.read_sql_query(
            "SELECT DISTINCT CAST(match_id AS TEXT) AS match_id "
            "FROM match_player_stats ORDER BY CAST(match_id AS TEXT) ASC", conn)

    matchhistory_df["match_id"] = matchhistory_df["match_id"].astype(str)
    matchhistory_df = matchhistory_df.drop_duplicates(subset=["match_id"], keep="last")

    if "winner" not in matchhistory_df.columns:
        matchhistory_df["winner"] = ""
    if "winner" in mm_cols:
        wm = pd.read_sql_query(
            "SELECT CAST(match_id AS TEXT) AS match_id, "
            "MAX(TRIM(COALESCE(winner, ''))) AS winner_map "
            "FROM match_maps GROUP BY CAST(match_id AS TEXT)", conn)
        matchhistory_df = matchhistory_df.merge(wm, on="match_id", how="left")
        matchhistory_df["winner"] = (
            matchhistory_df["winner"].astype(str).str.strip().replace("nan", "")
            .where(matchhistory_df["winner"].astype(str).str.strip() != "",
                   matchhistory_df["winner_map"])
        )
        matchhistory_df.drop(columns=["winner_map"], inplace=True)

    playerlist_df = pd.read_sql_query(f"""
        SELECT DISTINCT
            CAST(mps.match_id AS TEXT) AS match_id,
            CAST(mps.{steam_col} AS TEXT) AS steamid,
            TRIM(mps.{team_col}) AS team_name,
            TRIM(mps.{name_col}) AS player_name
        FROM match_player_stats mps
        WHERE mps.{steam_col} IS NOT NULL
          AND TRIM(COALESCE(mps.{team_col}, '')) <> ''
          AND TRIM(COALESCE(mps.{name_col}, '')) <> ''
        ORDER BY CAST(mps.match_id AS TEXT) ASC, team_name, player_name
    """, conn)
    playerlist_df = playerlist_df.drop_duplicates(subset=["match_id", "steamid"], keep="last")

    player_damage_df = pd.read_sql_query(f"""
        SELECT CAST(match_id AS TEXT) AS match_id,
               CAST({steam_col} AS TEXT) AS steamid,
               SUM(COALESCE(damage, 0)) AS total_damage
        FROM match_player_stats
        GROUP BY CAST(match_id AS TEXT), CAST({steam_col} AS TEXT)
    """, conn)

    rounds_df = pd.read_sql_query("""
        SELECT CAST(match_id AS TEXT) AS match_id,
               SUM(COALESCE(team1_score, 0) + COALESCE(team2_score, 0)) AS total_rounds
        FROM match_maps GROUP BY CAST(match_id AS TEXT)
    """, conn)

# Build outcome table 
player_match_outcome_df = playerlist_df.merge(
    matchhistory_df[["match_id", "winner"]], on="match_id", how="left")

team_map_by_match = {
    mid: _build_team_mapping_for_match(g)
    for mid, g in player_match_outcome_df.groupby("match_id", sort=False)
}
player_match_outcome_df["team_name"] = player_match_outcome_df.apply(
    lambda r: _map_team_name(r["match_id"], r["team_name"], team_map_by_match), axis=1)
player_match_outcome_df["winner"] = player_match_outcome_df.apply(
    lambda r: _map_winner_name(r["match_id"], r["winner"], team_map_by_match), axis=1)

def _result_label(row):
    winner = str(row.get("winner") or "").strip()
    team = str(row.get("team_name") or "").strip()
    if not winner or not team or team.lower() == "all":
        return "unknown"
    return "win" if team == winner else "loose"

player_match_outcome_df["result"] = player_match_outcome_df.apply(_result_label, axis=1)

# Build ADR table 
player_match_adr_df = (
    player_match_outcome_df[["match_id", "steamid", "player_name"]]
    .drop_duplicates(subset=["match_id", "steamid"], keep="first")
    .merge(player_damage_df, on=["match_id", "steamid"], how="left")
    .merge(rounds_df, on="match_id", how="left")
)
player_match_adr_df["total_damage"] = pd.to_numeric(
    player_match_adr_df["total_damage"], errors="coerce").fillna(0.0)
player_match_adr_df["total_rounds"] = pd.to_numeric(
    player_match_adr_df["total_rounds"], errors="coerce")
player_match_adr_df["adr"] = (
    player_match_adr_df["total_damage"]
    / player_match_adr_df["total_rounds"].where(player_match_adr_df["total_rounds"] > 0)
)

known = (player_match_outcome_df["result"].isin(["win", "loose"])).sum()
print(f"Data loaded: {len(player_match_outcome_df)} outcome rows "
      f"({known} with known result), {len(player_match_adr_df)} ADR rows")

Data loaded: 125 outcome rows (124 with known result), 125 ADR rows


In [30]:
# Model function (reusable with any tune dict) 

def _split_two_teams(match_df):
    team_rows = []
    for team_name, g in match_df.groupby("team_name", sort=False):
        if str(team_name).lower() == "all":
            continue
        outcomes = g["result"].dropna().unique().tolist()
        if not outcomes:
            continue
        outcome = outcomes[0]
        if outcome not in {"win", "loose"}:
            continue
        team_rows.append((team_name, outcome, g.copy()))
    if len(team_rows) != 2:
        return None
    return team_rows[0], team_rows[1]


def run_elo(outcome_df, adr_df, tune):
    """Run strict-timeline ADR-Elo.

    Returns (history_df, summary_df, final_global_anchor).
    Accepts any tune dict — missing keys fall back to TUNE defaults.
    """
    t = {**TUNE, **tune}
    k_factor    = float(t["K_FACTOR"])
    base_rating = float(t["BASE_RATING"])
    alpha       = float(t["ADR_ALPHA"])
    spread      = float(t["ADR_SPREAD"])
    min_mult    = float(t["ADR_MIN_MULT"])
    max_mult    = float(t["ADR_MAX_MULT"])
    prior_matches       = float(t["ADR_PRIOR_MATCHES"])
    initial_global_anchor = float(t["INITIAL_GLOBAL_ANCHOR"])

    work_df = outcome_df.copy()
    work_df["match_id"] = work_df["match_id"].astype(str)
    work_df = work_df[work_df["result"].isin(["win", "loose"])].copy()
    work_df = work_df.sort_values(["match_id", "steamid", "team_name"], kind="stable")
    work_df = work_df.drop_duplicates(subset=["match_id", "steamid"], keep="first")

    adr_lu = adr_df[["match_id", "steamid", "adr"]].copy()
    adr_lu["match_id"] = adr_lu["match_id"].astype(str)
    adr_lu["steamid"]  = adr_lu["steamid"].astype(str)
    work_df["steamid"] = work_df["steamid"].astype(str)
    work_df = work_df.merge(adr_lu, on=["match_id", "steamid"], how="left")

    ordered = sorted(work_df["match_id"].dropna().unique().tolist(), key=_match_sort_key)
    ratings, rows = {}, []
    player_adr_sum, player_adr_count = {}, {}
    global_adr_sum, global_adr_count = 0.0, 0

    for match_id in ordered:
        mdf = work_df[work_df["match_id"] == match_id]
        split = _split_two_teams(mdf)
        if split is None:
            continue

        (tn_a, res_a, df_a), (tn_b, res_b, df_b) = split
        sc_a = 1.0 if res_a == "win" else 0.0
        sc_b = 1.0 - sc_a

        for sid in pd.concat([df_a["steamid"], df_b["steamid"]]).astype(str).tolist():
            ratings.setdefault(sid, base_rating)

        ids_a = df_a["steamid"].astype(str).tolist()
        ids_b = df_b["steamid"].astype(str).tolist()
        elo_a = sum(ratings[s] for s in ids_a) / max(1, len(ids_a))
        elo_b = sum(ratings[s] for s in ids_b) / max(1, len(ids_b))

        exp_a = 1.0 / (1.0 + 10.0 ** ((elo_b - elo_a) / 400.0))
        exp_b = 1.0 - exp_a
        delta_a = k_factor * (sc_a - exp_a)
        delta_b = k_factor * (sc_b - exp_b)

        for tn, tdf, res, sc, exp_sc, t_elo, o_elo, t_delta in [
            (tn_a, df_a, res_a, sc_a, exp_a, elo_a, elo_b, delta_a),
            (tn_b, df_b, res_b, sc_b, exp_b, elo_b, elo_a, delta_b),
        ]:
            for _, row in tdf.iterrows():
                sid = str(row["steamid"])
                anchor = global_adr_sum / global_adr_count if global_adr_count > 0 else initial_global_anchor
                p_cnt = int(player_adr_count.get(sid, 0))
                p_sum = float(player_adr_sum.get(sid, 0.0))
                expected_adr = ((p_sum / p_cnt) * p_cnt + anchor * prior_matches) / (p_cnt + prior_matches) if p_cnt > 0 else anchor

                adr_val = pd.to_numeric(row.get("adr"), errors="coerce")
                if pd.isna(adr_val):
                    adr_val = expected_adr
                adr_z = (float(adr_val) - float(expected_adr)) / spread
                mult = _clip(1.0 + alpha * adr_z, min_mult, max_mult)

                elo_before = ratings[sid]
                elo_delta = t_delta * mult
                ratings[sid] += elo_delta

                rows.append({
                    "match_id": match_id, "steamid": sid,
                    "player_name": str(row["player_name"]),
                    "team_name": tn, "result": res,
                    "elo_before": elo_before, "elo_after": ratings[sid],
                    "elo_delta": elo_delta, "adr": float(adr_val),
                    "adr_expected": float(expected_adr),
                    "adr_multiplier": mult,
                    "team_elo_before": t_elo,
                    "opponent_team_elo_before": o_elo,
                    "global_anchor_used": anchor,
                })

        combined = pd.concat([df_a, df_b], ignore_index=True)[["steamid", "adr"]].copy()
        combined["adr"] = pd.to_numeric(combined["adr"], errors="coerce")
        for _, urow in combined.dropna(subset=["adr"]).iterrows():
            sid = str(urow["steamid"])
            obs = float(urow["adr"])
            player_adr_sum[sid]   = player_adr_sum.get(sid, 0.0) + obs
            player_adr_count[sid] = player_adr_count.get(sid, 0) + 1
            global_adr_sum += obs
            global_adr_count += 1

    history_df = pd.DataFrame(rows)
    if history_df.empty:
        raise RuntimeError("No Elo rows computed.")

    summary_df = (
        history_df
        .sort_values(["steamid", "match_id"],
                     key=lambda s: s.map(_match_sort_key), kind="stable")
        .groupby("steamid", as_index=False)
        .agg(
            name=("player_name", "last"),
            matches_played=("match_id", "nunique"),
            wins=("result", lambda s: int((s == "win").sum())),
            losses=("result", lambda s: int((s == "loose").sum())),
            adr_mean=("adr", "mean"),
            elo=("elo_after", "last"),
        )
        .sort_values("elo", ascending=False)
        .reset_index(drop=True)
    )
    summary_df["winrate"] = ((summary_df["wins"] / summary_df["matches_played"]) * 100).round(1)
    summary_df["adr_mean"] = summary_df["adr_mean"].round(1)
    summary_df["elo"] = summary_df["elo"].round(1)

    final_anchor = global_adr_sum / global_adr_count if global_adr_count > 0 else initial_global_anchor
    return history_df, summary_df, final_anchor

print("Model function ready: run_elo(outcome_df, adr_df, tune)")

Model function ready: run_elo(outcome_df, adr_df, tune)


In [31]:
#  Baseline run 
history_df, summary_df, final_anchor = run_elo(
    player_match_outcome_df, player_match_adr_df, TUNE)

print(f"Baseline: {len(summary_df)} players, "
      f"{history_df['match_id'].nunique()} matches, "
      f"anchor {final_anchor:.1f}")

# 1. Final Elo ranking bar chart 
fig_rank = px.bar(
    summary_df.sort_values("elo"),
    x="elo", y="name", orientation="h",
    color="winrate",
    color_continuous_scale="RdYlGn",
    range_color=[30, 70],
    text="elo",
    labels={"elo": "Elo Rating", "name": "Player", "winrate": "Win %"},
    title="Player Elo Ratings (baseline K=32, α=0.35)",
    height=max(400, len(summary_df) * 32),
)
fig_rank.update_traces(texttemplate="%{text:.0f}", textposition="outside")
fig_rank.update_layout(yaxis_title=None, xaxis_title="Elo",
                       margin=dict(l=120))
fig_rank.show()

# 2. Elo progression over matches 
hist = history_df.copy()
hist["match_idx"] = hist.groupby("steamid").cumcount() + 1
top_sids = summary_df.head(10)["steamid"].tolist()
top_hist = hist[hist["steamid"].isin(top_sids)]

fig_prog = px.line(
    top_hist, x="match_idx", y="elo_after",
    color="player_name", markers=True,
    labels={"match_idx": "Match #", "elo_after": "Elo", "player_name": "Player"},
    title="Elo Progression — Top 10 Players",
)
fig_prog.update_layout(height=450)
fig_prog.show()

# 3. ADR multiplier distribution 
fig_mult = px.histogram(
    history_df, x="adr_multiplier", nbins=30,
    color_discrete_sequence=["#636EFA"],
    labels={"adr_multiplier": "ADR Multiplier"},
    title="Distribution of ADR Multipliers (baseline)",
)
fig_mult.add_vline(x=1.0, line_dash="dash", line_color="red",
                   annotation_text="neutral (1.0)")
fig_mult.update_layout(height=350)
fig_mult.show()

Baseline: 23 players, 12 matches, anchor 79.5


## Parameter Sensitivity Analysis

Sweep key parameters while holding others at their baseline values to see how each knob shapes the rating distribution and player spread.

In [32]:
# K-factor sweep 
K_VALUES = [16, 24, 32, 40, 48, 64]

k_summaries = []
k_histories = []
for k_val in K_VALUES:
    h, s, _ = run_elo(player_match_outcome_df, player_match_adr_df, {"K_FACTOR": k_val})
    s = s.copy()
    s["K"] = k_val
    k_summaries.append(s)
    h = h.copy()
    h["K"] = k_val
    h["match_idx"] = h.groupby("steamid").cumcount() + 1
    k_histories.append(h)

k_summary_all = pd.concat(k_summaries, ignore_index=True)
k_history_all = pd.concat(k_histories, ignore_index=True)

#  Chart: Elo spread vs K 
spread_stats = (
    k_summary_all.groupby("K")["elo"]
    .agg(["std", "min", "max", "median"])
    .reset_index()
)
spread_stats["range"] = spread_stats["max"] - spread_stats["min"]

fig_k_spread = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Elo Std Dev by K-Factor", "Elo Range (max − min) by K-Factor"],
)
fig_k_spread.add_trace(
    go.Bar(x=spread_stats["K"], y=spread_stats["std"],
           marker_color="#636EFA", name="Std Dev",
           text=spread_stats["std"].round(1), textposition="outside"),
    row=1, col=1,
)
fig_k_spread.add_trace(
    go.Bar(x=spread_stats["K"], y=spread_stats["range"],
           marker_color="#EF553B", name="Range",
           text=spread_stats["range"].round(1), textposition="outside"),
    row=1, col=2,
)
fig_k_spread.update_layout(
    title_text="How K-Factor Affects Rating Spread",
    height=400, showlegend=False,
)
fig_k_spread.update_xaxes(title_text="K-Factor")
fig_k_spread.update_yaxes(title_text="Elo points")
fig_k_spread.show()

#  Chart: Per-player Elo curves for selected K values 
show_k = [16, 32, 64]
top5 = summary_df.head(5)["steamid"].tolist()
k_top = k_history_all[
    (k_history_all["steamid"].isin(top5)) & (k_history_all["K"].isin(show_k))
].copy()
k_top["label"] = k_top["player_name"] + " (K=" + k_top["K"].astype(str) + ")"

fig_k_curves = px.line(
    k_top, x="match_idx", y="elo_after", color="label",
    line_dash="K", line_dash_map={16: "dot", 32: "solid", 64: "dash"},
    markers=False,
    labels={"match_idx": "Match #", "elo_after": "Elo", "label": ""},
    title="Elo Curves — Top 5 Players at K=16 / 32 / 64",
)
fig_k_curves.update_layout(height=500)
fig_k_curves.show()

#  Chart: Final ranking comparison 
rank_pivot = k_summary_all.pivot(index="name", columns="K", values="elo")
rank_pivot = rank_pivot.loc[rank_pivot[32].sort_values(ascending=False).index]

fig_k_rank = go.Figure()
colors = px.colors.qualitative.Set2
for i, k_val in enumerate(K_VALUES):
    fig_k_rank.add_trace(go.Scatter(
        x=rank_pivot[k_val], y=rank_pivot.index,
        mode="markers", name=f"K={k_val}",
        marker=dict(size=9, color=colors[i % len(colors)]),
    ))
fig_k_rank.update_layout(
    title="Final Elo per Player across K-Factor values",
    xaxis_title="Elo", height=max(400, len(rank_pivot) * 30),
    margin=dict(l=120),
)
fig_k_rank.show()

In [33]:
# ADR parameter sweep 
# Vary alpha (ADR influence strength) and spread (z-score denominator)
ALPHA_VALUES  = [0.0, 0.15, 0.25, 0.35, 0.45, 0.60]
SPREAD_VALUES = [8, 12, 15, 20, 30]

# . Alpha sweep (spread fixed at baseline) 
alpha_summaries = []
alpha_histories = []
for a in ALPHA_VALUES:
    h, s, _ = run_elo(player_match_outcome_df, player_match_adr_df, {"ADR_ALPHA": a})
    s = s.copy()
    s["alpha"] = a
    alpha_summaries.append(s)
    h = h.copy()
    h["alpha"] = a
    alpha_histories.append(h)

alpha_all = pd.concat(alpha_summaries, ignore_index=True)
alpha_hist_all = pd.concat(alpha_histories, ignore_index=True)

# Box plot: Elo distribution per alpha
fig_alpha_box = px.box(
    alpha_all, x="alpha", y="elo", points="all",
    color="alpha",
    labels={"alpha": "ADR Alpha (α)", "elo": "Final Elo"},
    title="Elo Distribution by ADR Alpha (K=32, spread=15)",
)
fig_alpha_box.update_layout(height=450, showlegend=False)
fig_alpha_box.show()

# Multiplier distributions: alpha=0 vs baseline vs strong
fig_mult_comp = make_subplots(
    rows=1, cols=3,
    subplot_titles=["α = 0.0 (pure Elo)", "α = 0.35 (baseline)", "α = 0.60 (strong)"],
)
for col_idx, a_val in enumerate([0.0, 0.35, 0.60]):
    subset = alpha_hist_all[alpha_hist_all["alpha"] == a_val]
    fig_mult_comp.add_trace(
        go.Histogram(x=subset["adr_multiplier"], nbinsx=25,
                     marker_color=["#00CC96", "#636EFA", "#EF553B"][col_idx],
                     name=f"α={a_val}"),
        row=1, col=col_idx + 1,
    )
    fig_mult_comp.add_vline(x=1.0, row=1, col=col_idx + 1,
                            line_dash="dash", line_color="gray")
fig_mult_comp.update_layout(
    title_text="ADR Multiplier Distributions at Different Alpha Values",
    height=350, showlegend=False,
)
fig_mult_comp.update_xaxes(title_text="Multiplier")
fig_mult_comp.show()

# . Spread sweep (alpha fixed at baseline) 
spread_summaries = []
for sp in SPREAD_VALUES:
    _, s, _ = run_elo(player_match_outcome_df, player_match_adr_df, {"ADR_SPREAD": sp})
    s = s.copy()
    s["spread"] = sp
    spread_summaries.append(s)

spread_all = pd.concat(spread_summaries, ignore_index=True)

fig_spread_box = px.box(
    spread_all, x="spread", y="elo", points="all",
    color="spread",
    labels={"spread": "ADR Spread", "elo": "Final Elo"},
    title="Elo Distribution by ADR Spread (K=32, α=0.35)",
)
fig_spread_box.update_layout(height=450, showlegend=False)
fig_spread_box.show()

# 3. Heatmap: Elo std vs (alpha, spread) 
heat_rows = []
for a in [0.0, 0.15, 0.25, 0.35, 0.45, 0.60]:
    for sp in [8, 12, 15, 20, 30]:
        _, s, _ = run_elo(player_match_outcome_df, player_match_adr_df,
                          {"ADR_ALPHA": a, "ADR_SPREAD": sp})
        heat_rows.append({"alpha": a, "spread": sp, "elo_std": s["elo"].std().round(1)})

heat_df = pd.DataFrame(heat_rows)
heat_pivot = heat_df.pivot(index="alpha", columns="spread", values="elo_std")

fig_heat = px.imshow(
    heat_pivot.values,
    x=[str(c) for c in heat_pivot.columns],
    y=[str(r) for r in heat_pivot.index],
    color_continuous_scale="YlOrRd",
    text_auto=".1f",
    labels={"x": "Spread", "y": "Alpha (α)", "color": "Elo Std Dev"},
    title="Rating Volatility: Elo Std Dev across (α, spread) combinations",
)
fig_heat.update_layout(height=400)
fig_heat.show()

print("Sensitivity analysis complete.")

Sensitivity analysis complete.


In [34]:
# quick numeric check: old baseline vs current tune vs one extra stability-first variant
configs = {
    "OLD  K32 a0.35 sp15": {"K_FACTOR": 32.0, "ADR_ALPHA": 0.35, "ADR_SPREAD": 15.0},
    "NEW  K24 a0.25 sp18": {"K_FACTOR": float(TUNE["K_FACTOR"]), "ADR_ALPHA": float(TUNE["ADR_ALPHA"]), "ADR_SPREAD": float(TUNE["ADR_SPREAD"])},
    "ALT  K24 a0.20 sp22": {"K_FACTOR": 24.0, "ADR_ALPHA": 0.20, "ADR_SPREAD": 22.0},
}

def _metrics(h, s):
    mult = h["adr_multiplier"]
    core = s[s["matches_played"] >= 3]
    return {
        "elo_std": float(s["elo"].std()),
        "elo_range": float(s["elo"].max() - s["elo"].min()),
        "core_std": float(core["elo"].std()) if len(core) > 1 else 0.0,
        "floor_pct": float((mult <= TUNE["ADR_MIN_MULT"]).mean() * 100.0),
        "ceil_pct": float((mult >= TUNE["ADR_MAX_MULT"]).mean() * 100.0),
        "near_neutral_pct": float(((mult >= 0.98) & (mult <= 1.02)).mean() * 100.0),
    }

results = {}
for label, ov in configs.items():
    h, s, _ = run_elo(player_match_outcome_df, player_match_adr_df, ov)
    results[label] = _metrics(h, s)

for label, vals in results.items():
    print(label, {k: round(v, 1) for k, v in vals.items()})

base = results["OLD  K32 a0.35 sp15"]
cur = results["NEW  K24 a0.25 sp18"]
print("\nDelta NEW-OLD:", {k: round(cur[k] - base[k], 1) for k in base})

OLD  K32 a0.35 sp15 {'elo_std': 39.1, 'elo_range': 173.4, 'core_std': 52.2, 'floor_pct': 50.0, 'ceil_pct': 36.3, 'near_neutral_pct': 3.2}
NEW  K24 a0.25 sp18 {'elo_std': 29.6, 'elo_range': 131.4, 'core_std': 39.6, 'floor_pct': 50.0, 'ceil_pct': 36.3, 'near_neutral_pct': 3.2}
ALT  K24 a0.20 sp22 {'elo_std': 30.1, 'elo_range': 134.9, 'core_std': 40.2, 'floor_pct': 25.8, 'ceil_pct': 25.8, 'near_neutral_pct': 4.8}

Delta NEW-OLD: {'elo_std': -9.5, 'elo_range': -42.0, 'core_std': -12.7, 'floor_pct': 0.0, 'ceil_pct': 0.0, 'near_neutral_pct': 0.0}
